## Шаг 1. Extract (Извлечение данных)

In [ ]:
!pip install "dask[complete]" graphviz pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 55.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 3.3 MB/s eta 0:00:00


In [ ]:
import dask.dataframe as dd
from dask.distributed import Client
from dask.diagnostics import ProgressBar
import dask.delayed as delayed
import pandas as pd
import numpy as np
import os
import json
from datetime import datetime

In [ ]:
# ПОСТРОЕНИЕ ETL-ПАЙПЛАЙНА

# Настройка локального кластера Dask
print("\nSetting local cluster Dask...")
client = Client(n_workers=2, threads_per_worker=2, processes=True)
display(client)


Setting local cluster Dask...


INFO:distributed.http.proxy:To route to workers diagnostics web server please install jupyter-server-proxy: python -m pip install jupyter-server-proxy
INFO:distributed.scheduler:State start
INFO:distributed.scheduler:  Scheduler at:     tcp://127.0.0.1:42299
INFO:distributed.scheduler:  dashboard at:  http://127.0.0.1:8787/status
INFO:distributed.scheduler:Registering Worker plugin shuffle
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:45717'
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:45275'
INFO:distributed.scheduler:Register worker addr: tcp://127.0.0.1:38243 name: 0
INFO:distributed.scheduler:Starting worker compute stream, tcp://127.0.0.1:38243
INFO:distributed.core:Starting established connection to tcp://127.0.0.1:51850
INFO:distributed.scheduler:Register worker addr: tcp://127.0.0.1:36443 name: 1
INFO:distributed.scheduler:Starting worker compute stream, tcp://127.0.0.1:36443
INFO:distributed.core:Starting established connection to tcp://127

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 2
Total threads: 4,Total memory: 12.67 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:42299,Workers: 0
Dashboard: http://127.0.0.1:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:38243,Total threads: 2
Dashboard: http://127.0.0.1:44161/status,Memory: 6.34 GiB
Nanny: tcp://127.0.0.1:45717,


## Extract - извлечение данных

In [ ]:
# Extract - ленивая загрузка данных
print("\nLoading dataset (Extract)...")

file_path = 'Parking_Violations_Issued_-_Fiscal_Year_2014__August_2013___June_2014_ (1).csv'

dtype_dict = {
    'Summons Number': 'int64',
    'Plate ID': 'object',
    'Registration State': 'object',
    'Plate Type': 'object',
    'Issue Date': 'object',
    'Violation Code': 'int64',
    'Vehicle Body Type': 'object',
    'Vehicle Make': 'object',
    'Issuing Agency': 'object',
    'Street Code1': 'float64',
    'Street Code2': 'float64',
    'Street Code3': 'float64',
    'Vehicle Expiration Date': 'object',
    'Violation Location': 'object',
    'Violation Precinct': 'float64',
    'Issuer Precinct': 'float64',
    'Issuer Code': 'float64',
    'Issuer Command': 'object',
    'Issuer Squad': 'object',
    'Violation Time': 'object',
    'Time First Observed': 'object',
    'Violation County': 'object',
    'Violation In Front Of Or Opposite': 'object',
    'House Number': 'object',
    'Street Name': 'object',
    'Intersecting Street': 'object',
    'Date First Observed': 'object',
    'Law Section': 'float64',
    'Sub Division': 'object',
    'Violation Legal Code': 'object',
    'Days Parking In Effect    ': 'object',
    'From Hours In Effect': 'object',
    'To Hours In Effect': 'object',
    'Vehicle Color': 'object',
    'Unregistered Vehicle?': 'object',
    'Vehicle Year': 'float64',
    'Meter Number': 'object',
    'Feet From Curb': 'float64',
    'Violation Post Code': 'object',
    'Violation Description': 'object',
    'No Standing or Stopping Violation': 'object',
    'Hydrant Violation': 'object',
    'Double Parking Violation': 'object',
    'Latitude': 'float64',
    'Longitude': 'float64',
    'Community Board': 'float64',
    'Community Council ': 'object',
    'Census Tract': 'float64',
    'BIN': 'float64',
    'BBL': 'float64',
    'NTA': 'object'
}

# Ленивая загрузка
df = dd.read_csv(file_path, dtype=dtype_dict, assume_missing=True)

# Структура датасета
print("= Dataset =")
print(f"Partitions quantity: {df.npartitions}")
print(f"Columns quantity: {len(df.columns)}")
print(f"List of columns:\n{list(df.columns)}")

# Показываем первые 5 строк (вызывает вычисления только для head)
print("\nFirst 5 rows:")
print(df.head())

print("\nData types:")
print(df.dtypes)


Loading dataset (Extract)...
= Dataset =
Partitions quantity: 1
Columns quantity: 51
List of columns:
['Summons Number', 'Plate ID', 'Registration State', 'Plate Type', 'Issue Date', 'Violation Code', 'Vehicle Body Type', 'Vehicle Make', 'Issuing Agency', 'Street Code1', 'Street Code2', 'Street Code3', 'Vehicle Expiration Date', 'Violation Location', 'Violation Precinct', 'Issuer Precinct', 'Issuer Code', 'Issuer Command', 'Issuer Squad', 'Violation Time', 'Time First Observed', 'Violation County', 'Violation In Front Of Or Opposite', 'House Number', 'Street Name', 'Intersecting Street', 'Date First Observed', 'Law Section', 'Sub Division', 'Violation Legal Code', 'Days Parking In Effect    ', 'From Hours In Effect', 'To Hours In Effect', 'Vehicle Color', 'Unregistered Vehicle?', 'Vehicle Year', 'Meter Number', 'Feet From Curb', 'Violation Post Code', 'Violation Description', 'No Standing or Stopping Violation', 'Hydrant Violation', 'Double Parking Violation', 'Latitude', 'Longitude',

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

## Transform - анализ качества данных

In [ ]:
# Кол-во пропусков
missing_values = df.isnull().sum()

# Кол-во строк
total_rows = df.shape[0]

# Процент пропусков
with ProgressBar():
    missing_counts = missing_values.compute()
    total_rows_count = total_rows.compute()

print(f"\nTotal rows count: {total_rows_count:,}")

# Словарь с процентами пропусков
missing_percent = {}
print("\nTop-20 of missing objects (топ-20 по пропускам):")
missing_with_percent = []

for col in missing_counts.index:
    percent = (missing_counts[col] / total_rows_count) * 100
    missing_percent[col] = percent
    if percent > 0:
        missing_with_percent.append((col, percent, missing_counts[col]))

# Сортировка по проценту пропусков и топ-20
missing_with_percent.sort(key=lambda x: x[1], reverse=True)
for col, percent, count in missing_with_percent[:20]:
    print(f"  {col}: {percent:.2f}% ({count:,} пропусков)")

print(f"\n*** Q-ty of columns with missing values: {len(missing_with_percent)}")


Total rows count: 265,107

Top-20 of missing objects (топ-20 по пропускам):
  No Standing or Stopping Violation: 100.00% (265,107 пропусков)
  Hydrant Violation: 100.00% (265,107 пропусков)
  Double Parking Violation: 100.00% (265,107 пропусков)
  Latitude: 100.00% (265,107 пропусков)
  Longitude: 100.00% (265,107 пропусков)
  Community Board: 100.00% (265,107 пропусков)
  Community Council : 100.00% (265,107 пропусков)
  Census Tract: 100.00% (265,107 пропусков)
  BIN: 100.00% (265,107 пропусков)
  BBL: 100.00% (265,107 пропусков)
  NTA: 100.00% (265,107 пропусков)
  Violation Legal Code: 90.14% (238,960 пропусков)
  Time First Observed: 89.57% (237,462 пропусков)
  Intersecting Street: 77.89% (206,482 пропусков)
  Unregistered Vehicle?: 76.67% (203,258 пропусков)
  Meter Number: 69.36% (183,878 пропусков)
  From Hours In Effect: 34.05% (90,256 пропусков)
  To Hours In Effect: 34.05% (90,256 пропусков)
  Violation Post Code: 33.09% (87,714 пропусков)
  Violation Description: 23.33% (

In [ ]:
# Порог для удаления столбцов в проц
THRESHOLD = 90

# Список столбцов для удаления
columns_to_drop = [col for col, percent in missing_percent.items() if percent > THRESHOLD]

print(f"Columns to drop (>{THRESHOLD}% missing): {len(columns_to_drop)}")
print(f"Columns to remove:\n{columns_to_drop}")

# Ленивое удаление столбцов
df_dropped = df.drop(columns=columns_to_drop)

print(f"\nColumns count after removal: {len(df_dropped.columns)}")
print(f"Remaining columns: {list(df_dropped.columns)}")

Columns to drop (>90% missing): 12
Columns to remove:
['Violation Legal Code', 'No Standing or Stopping Violation', 'Hydrant Violation', 'Double Parking Violation', 'Latitude', 'Longitude', 'Community Board', 'Community Council ', 'Census Tract', 'BIN', 'BBL', 'NTA']

Columns count after removal: 39
Remaining columns: ['Summons Number', 'Plate ID', 'Registration State', 'Plate Type', 'Issue Date', 'Violation Code', 'Vehicle Body Type', 'Vehicle Make', 'Issuing Agency', 'Street Code1', 'Street Code2', 'Street Code3', 'Vehicle Expiration Date', 'Violation Location', 'Violation Precinct', 'Issuer Precinct', 'Issuer Code', 'Issuer Command', 'Issuer Squad', 'Violation Time', 'Time First Observed', 'Violation County', 'Violation In Front Of Or Opposite', 'House Number', 'Street Name', 'Intersecting Street', 'Date First Observed', 'Law Section', 'Sub Division', 'Days Parking In Effect    ', 'From Hours In Effect', 'To Hours In Effect', 'Vehicle Color', 'Unregistered Vehicle?', 'Vehicle Year',

In [ ]:
# Преобразуем Issue Date с автоматическим определением формата и обработкой ошибок
if 'Issue Date' in df_dropped.columns:
    print("Converting Issue Date...")
    df_dropped['Issue Date'] = dd.to_datetime(
        df_dropped['Issue Date'],
        errors='coerce'  # Некорректные даты станут NaT
    )

# Преобразуем Vehicle Expiration Date
if 'Vehicle Expiration Date' in df_dropped.columns:
    print("Converting Vehicle Expiration Date...")
    df_dropped['Vehicle Expiration Date'] = dd.to_datetime(
        df_dropped['Vehicle Expiration Date'],
        errors='coerce'
    )

print("Date conversion completed")

Converting Issue Date...
Converting Vehicle Expiration Date...
Date conversion completed


In [ ]:
# Удалил дубликаты
if 'Summons Number' in df_dropped.columns:
    print("Removing duplicates by Summons Number...")
    df_cleaned = df_dropped.drop_duplicates(subset=['Summons Number'])
else:
    df_cleaned = df_dropped

# Аномалка значений Vehicle Year
if 'Vehicle Year' in df_cleaned.columns:
    print("Filtering innormal vehicle years...")
    df_cleaned = df_cleaned[(df_cleaned['Vehicle Year'] >= 1900) |
                             (df_cleaned['Vehicle Year'].isna())]
    df_cleaned = df_cleaned[(df_cleaned['Vehicle Year'] <= 2026) |
                             (df_cleaned['Vehicle Year'].isna())]

print("Cleaning string columns...")

# Очистка строк
def clean_string(s):
    """Очищает строку: убирает пробелы, приводит к верхнему регистру"""
    if isinstance(s, str):
        return s.strip().upper()
    return s

string_columns = ['Vehicle Make', 'Violation County', 'Plate ID', 'Registration State',
                  'Plate Type', 'Vehicle Body Type', 'Issuing Agency', 'Street Name',
                  'Violation Description', 'Vehicle Color']

for col in string_columns:
    if col in df_cleaned.columns:
        print(f"- - - Cleaning {col}...")
        df_cleaned[col] = df_cleaned[col].map(clean_string, meta=(col, 'object'))

print("Data cleaning completed")

Removing duplicates by Summons Number...
Filtering innormal vehicle years...
Cleaning string columns...
- - - Cleaning Vehicle Make...
- - - Cleaning Violation County...
- - - Cleaning Plate ID...
- - - Cleaning Registration State...
- - - Cleaning Plate Type...
- - - Cleaning Vehicle Body Type...
- - - Cleaning Issuing Agency...
- - - Cleaning Street Name...
- - - Cleaning Violation Description...
- - - Cleaning Vehicle Color...
Data cleaning completed


In [ ]:
print("Statistics analysing...")
with ProgressBar():
    # Количество уникальных штрафов
    if 'Summons Number' in df_cleaned.columns:
        unique_count = df_cleaned['Summons Number'].nunique().compute()
        print(f"Unique summons count: {unique_count:,}")

    # Топ-5 марок автомобилей
    if 'Vehicle Make' in df_cleaned.columns:
        # Убираем пустые значения
        vehicle_makes = df_cleaned['Vehicle Make'].dropna()
        top_vehicles = vehicle_makes.value_counts().nlargest(10).compute()
        print(f"\nTop-10 vehicle makes:")
        for idx, (make, count) in enumerate(top_vehicles.items(), 1):
            if make and make != '' and make != 'NONE':
                print(f"  {idx}. {make}: {count:,}")

    # Топ-5 округов по нарушениям
    if 'Violation County' in df_cleaned.columns:
        # Убираем пустые значения и приводим к единому формату
        counties = df_cleaned['Violation County'].dropna()
        # Объединяем BX и BRONX в один округ
        counties = counties.map(lambda x: 'BRONX' if x in ['BX', 'BRONX'] else x)
        top_counties = counties.value_counts().nlargest(5).compute()
        print(f"\nTop-5 counties by violations:")
        county_names = {
            'NY': 'MANHATTAN',
            'K': 'BROOKLYN',
            'Q': 'QUEENS',
            'BRONX': 'BRONX',
            'R': 'STATEN ISLAND'
        }
        for idx, (county, count) in enumerate(top_counties.items(), 1):
            display_name = county_names.get(county, county)
            print(f"  {idx}. {display_name}: {count:,}")

    # Статистика по годам
    if 'Issue Year' in df_cleaned.columns:
        years_stats = df_cleaned['Issue Year'].dropna().value_counts().nlargest(5).compute()
        print(f"\nTop-5 years with most violations:")
        for idx, (year, count) in enumerate(years_stats.items(), 1):
            print(f"  {idx}. {int(year)}: {count:,}")

print("\n- Statistics done.")

Statistics analysing...


Unique summons count: 178,125


/usr/local/lib/python3.12/dist-packages/dask/dataframe/dask_expr/_collection.py:4218: UserWarning: 
You did not provide metadata, so Dask is running your function on a small dataset to guess output types. It is possible that Dask will guess incorrectly.
To provide an explicit output types or to silence this message, please provide the `meta=` keyword, as described in the map function that you are using.
  Before: .map(func)
  After:  .map(func, meta=('Violation County', 'object'))

  warnings.warn(meta_warning(meta, method="map"))



Top-10 vehicle makes:
  1. FORD: 29,656
  2. CHEVR: 17,013
  3. TOYOT: 15,578
  4. HONDA: 13,210
  5. NISSA: 11,409
  6. GMC: 8,819
  7. INTER: 6,442
  8. DODGE: 6,193
  9. LINCO: 5,944
  10. FRUEH: 4,775



Top-5 counties by violations:
  1. MANHATTAN: 54,772
  2. BROOKLYN: 38,565
  3. QUEENS: 36,832
  4. BRONX: 19,434
  5. STATEN ISLAND: 1,657

- Statistics done.


## LOAD - сохранение результатов

In [ ]:
print("===== SAVING =====")

# Сохраняем очищенные данные в формате Parquet
output_parquet = 'cleaned_dataset.parquet'
print(f"Saving to Parquet: {output_parquet}")

if 'Issue Date' in df_cleaned.columns:
    # Преобразуем datetime обратно в строку для сохранения
    df_cleaned['Issue Date'] = df_cleaned['Issue Date'].astype(str)

if 'Vehicle Expiration Date' in df_cleaned.columns:
    df_cleaned['Vehicle Expiration Date'] = df_cleaned['Vehicle Expiration Date'].astype(str)

# Сохраняем в Parquet
df_cleaned.to_parquet(output_parquet, engine='pyarrow', write_index=False)
print("✓ Parquet saved")

# Cтатистика в JSON
print("Saving statistics...")
statistics = {
    'total_records_original': total_rows_count,
    'total_columns_original': len(df.columns),
    'total_columns_after_dropping': len(df_dropped.columns),
    'total_columns_final': len(df_cleaned.columns),
    'columns_removed': columns_to_drop,
    'missing_percentages': {k: round(v, 2) for k, v in missing_percent.items() if v > 0}
}

with open('statistics.json', 'w', encoding='utf-8') as f:
    json.dump(statistics, f, indent=2, ensure_ascii=False, default=str)

print("[DONE] Statistis saved to statistics.json")

print("\n[DONE} ETL PIPELINE COMPLETED SUCCESSFULLY")

===== SAVING =====
Saving to Parquet: cleaned_dataset.parquet


✓ Parquet saved
Saving statistics...
[DONE] Statistis saved to statistics.json

[DONE} ETL PIPELINE COMPLETED SUCCESSFULLY


## Визуализация DAG

In [13]:
print("\n\nSIMPLE DAG VISUALIZATION")

# Определяем функции для вычислений
def increment(x):
    return x + 1

def double(x):
    return x * 2

def add(x, y):
    return x + y

# Создание отложенных объектов
x = delayed(increment)(5)
y = delayed(double)(x)
z = delayed(increment)(10)
result = delayed(add)(y, z)

# Визуализация графа
print("Creating simple_dag.png...")
result.visualize(filename='simple_dag', format='png')
print("[DONE] simple_dag.png created")

# Выполняем вычисления
final_result = result.compute()
print(f"Computation result: 5 -> +1=6, *2=12, + (10+1=11) = {final_result}")



SIMPLE DAG VISUALIZATION
Creating simple_dag.png...
[DONE] simple_dag.png created
Computation result: 5 -> +1=6, *2=12, + (10+1=11) = 23


In [14]:
print("=== MAP-REDUCE DAG VISUALIZATION ===")

# Функция для map
def square(x):
    """Возводит число в квадрат"""
    return x * x

# Функция для reduce
def sum_all(numbers):
    """Суммирует все элементы списка"""
    return sum(numbers)

# Исходные данные
data = [1, 2, 3, 4, 5]

# Map - применяем square к каждому элементу
mapped = [delayed(square)(i) for i in data]

# Reduce - суммируем все результаты
total = delayed(sum_all)(mapped)

# Визуализируем граф
print("Creating mapreduce_dag.png visualization...")
total.visualize(filename='mapreduce_dag', format='png')
print("[DONE] mapreduce_dag.png created")

# Выполняем вычисления
result = total.compute()
print(f"Sum of squares (1²+2²+3²+4²+5²) = {result}")

=== MAP-REDUCE DAG VISUALIZATION ===
Creating mapreduce_dag.png visualization...
[DONE] mapreduce_dag.png created
Sum of squares (1²+2²+3²+4²+5²) = 55
